In [2]:
# # 这里采用xgboost算法进行预测
# import numpy as np
# import pandas as pd
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import r2_score, mean_absolute_percentage_error, mean_absolute_error
# from xgboost import XGBRegressor
# from utils import create_dir
# import joblib
# import os
# 
# # 配置路径和选择
# model_name = 'xgboost'
# save_model = True
# del_0qf = True  # 是否删除0屈服强度数据
# main_path = f'models/{model_name}'  # 这里是模型所在的文件位置
# create_dir(main_path, is_mainpath=True)
# excel = 'index/xgboost_index.xlsx'
# if not os.path.exists(excel):
#     pd.DataFrame().to_excel(excel)
# 
# # 检查分数表格是否存在，如果不存在就新建一个
# score_path = 'score_different_algorithms.xlsx'
# if not os.path.exists(score_path):
#     df = pd.DataFrame(columns=['Model', 'R^2', 'MAPE', 'MAE'])
#     df.to_excel(score_path, index=False)
# 
# # 文件读取部分
# df = pd.read_excel('train_set.xlsx', index_col=0)  # 读取 Excel 文件
# # 删除包含空值的行
# 
# np.random.seed(200)  # 设置随机数种子
# # 提取特征和目标变量
# feature_names = df.drop(columns=['Precipitate Distribution', '屈服强度', '抗拉强度 (UTS)', '追踪编号']).columns
# X = df[feature_names]  # 特征: 最后四列之前的所有列
# y = df['抗拉强度 (UTS)']  # 目标: 倒数第四列
# 
# # 训练模型
# model = XGBRegressor(random_state=200)
# model.fit(X, y)
# 
# # 在整个数据集上预测
# y_pred = model.predict(X)
# 
# # 计算误差
# errors = np.abs(y_pred - y)
# # 找到误差最大的30个样本的索引
# top_30_error_indices = errors.nlargest(30).index
# 
# # 从整个数据集中剔除这30个样本
# df_new = df.drop(index=top_30_error_indices)
# 
# # 将剩下的样本保存到新的train_set_new.xlsx中
# df_new.to_excel('train_set_new.xlsx')
# 
# # 在新的数据集上训练模型并评估性能
# X_new = df_new[feature_names]
# y_new = df_new['抗拉强度 (UTS)']
# 
# # 重新训练模型
# model.fit(X_new, y_new)
# y_new_pred = model.predict(X_new)
# 
# # 计算模型评估指标
# r2_new = r2_score(y_new, y_new_pred)
# mae_new = mean_absolute_error(y_new, y_new_pred)
# mape_new = mean_absolute_percentage_error(y_new, y_new_pred)
# 
# print(f"R2: {r2_new}")
# print(f"MAE: {mae_new}")
# print(f"MAPE: {mape_new}")


In [3]:
# 这里采用xgboost算法进行预测
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold,train_test_split,cross_validate,cross_val_score,KFold
from sklearn.metrics import r2_score, mean_absolute_percentage_error,mean_absolute_error,make_scorer
from xgboost import XGBRegressor
from utils import create_empty_ls,create_dir,mycopyfile,mycopydir
import joblib
from openpyxl import load_workbook
import os
"""
路径的配置和选择
"""
model_name='xgboost'

save_model=True
del_0qf=True #是否删除0屈服强度数据
main_path=f'models/{model_name}' # 这里是模型所在的文件位置
create_dir(main_path,is_mainpath=True)
excel='index/xgboost_index.xlsx'
if not os.path.exists(excel):
    pd.DataFrame().to_excel(excel)
# 如果分数表格不存在就新建一个
score_path='score_different_algorithms.xlsx'
if os.path.exists(score_path):
    print(f"The file '{score_path}' exists.")
else:
    print(f"The file '{score_path}' does not exist.")
    df = pd.DataFrame(columns=['Model', 'R^2', 'MAPE', 'MAE'])
    df.to_excel('score_different_algorithms.xlsx', index=False)

"""
文件读取部分
"""
# 读取 Excel 文件
df = pd.read_excel('train_set_new.xlsx', index_col=0)  # 引入这一列之后，原本的第一列就是一个索引，第0列会从有意义的列开始
# 删除包含空值的行

np.random.seed(200) #设置随机数种子
feature_names = df.drop(columns=['Precipitate Distribution', '屈服强度', '抗拉强度 (UTS)', '追踪编号']).columns 
print(feature_names)
X = df.drop(columns=['Precipitate Distribution', '屈服强度', '抗拉强度 (UTS)', '追踪编号'])  # 特征: 最后四列之前的所有列
y = df.iloc[:, -2]  # 目标: 倒数第四列
# 将目标变量分箱为10个类别
bins = np.linspace(y.min(), y.max(), 11)
y_binned = np.digitize(y, bins)
# print('y_binned',y_binned)
# 这里是随机划分数据集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=200)

"""
设置初始化
"""
n_splits=5
# 初始化StratifiedKFold
# 初始化XGBoost回归器
# model = XGBRegressor(random_state=200,gamma=0.2)# 设置随机种子以确保结果的可重复性
# 初始化列表以存储每个折叠的分数
# 定义评估指标
scoring = {
    'r2':make_scorer(r2_score,greater_is_better=True),
    'MAE': make_scorer(mean_absolute_error, greater_is_better=False),
    'MAPE': make_scorer(mean_absolute_percentage_error, greater_is_better=False)
}
r2_scores_ls = []
mape_scores = []
mae_scores=[]
best_epoch=0
best_r2_score = float('-inf')
best_mae_score= float('-inf')
best_mape_score= float('-inf')
ls1=['fold'+str(i) for i in range(1,n_splits+1)]
fold_dict=dict(zip(ls1,[0 for i in range(0,len(ls1))]))
num_each_fold_train=np.floor((np.int32(X.shape[0])-1)*(n_splits-1)/n_splits)
num_each_fold_test=(np.int32(X.shape[0])-1)//n_splits

"""
模型训练部分,根据交叉验证结果评估模型性能
"""
epoches=1
for epoch in range(epoches):
    kf = KFold(n_splits=5, shuffle=True, random_state=60)# 注意这里修改了random_state以确保每次初始化不同
    model = XGBRegressor(random_state=200, gamma=0.2)  # 注意这里修改了random_state以确保每次初始化不同
    model.fit(X_train, y_train)
    
    # 计算分折交叉验证结果
    results = cross_validate(model,X,y,cv=kf, scoring=scoring, return_train_score=False)
    # 输出分折交叉验证结果
    for i, (r2, mae, mape) in enumerate(zip(results['test_r2'],results['test_MAE'], results['test_MAPE']), start=1):
        # 记录不同折的分数结果
        r2_scores_ls.append(r2)
        mape_scores.append(-mape)
        mae_scores.append(-mae)
        # print(f"折 {i}: r2={r2:.3f},MAE = {-mae:.3f}, MAPE = {-mape:.3f}")

    current_r2_score=np.mean(r2_scores_ls)
    # scores = cross_val_score(model, X, y_binned, cv=skf, scoring=make_scorer(r2_score))
    # print('r2_scores:',scores)
    # current_r2_score=scores.mean()
    # 只对更优的结果进行保存
    if current_r2_score>best_r2_score:
        best_epoch=epoch
        # 记录模型的特征重要性
        feature_importance_final=model.feature_importances_
        importance_dict = dict(zip(feature_importance_final, feature_names))
        importance_tuplelist = [(abs(value), feature) for (value, feature) in importance_dict.items()]
        importance_sorted = sorted(importance_tuplelist, reverse=True)
        x_feature = [j for (i, j) in importance_sorted]
        y_importance_value = [i for (i, j) in importance_sorted]
        print(f'{model_name}_tree_feature_names(top15):')
        for i in x_feature[:15]:
            print(i)
        print(f'average_{model_name}_feature_values(top15):')
        for i in y_importance_value[:15]:
            print(i)
        
        #记录更优的分数
        best_r2_score=current_r2_score
        best_mae_score=np.mean(mae_scores)
        best_mape_score=np.mean(mape_scores)
        if save_model:
            joblib.dump(model,   f'models/{model_name}/{model_name}_best_new.pkl')
            print(f'保存更优{model_name}模型文件') 
        print('平均 R2 分数:', best_r2_score)
        print('平均 MAPE 分数:', best_mape_score * 100)  # 转换为百分比
        print('平均MAE分数:',best_mae_score)
"""
打印并输出分数至表格
"""

df = pd.read_excel('score_different_algorithms.xlsx')
# 添加新数据的示例
new_data = {'Model': f'{model_name}', 'R^2': best_r2_score , 'MAPE': best_mape_score * 100, 'MAE': best_mae_score}
new_data_df = pd.DataFrame([new_data])
# 将新数据添加到数据框中
df = pd.concat([df, new_data_df], ignore_index=True)
# 保存数据框到Excel表格
df.to_excel('score_different_algorithms.xlsx', index=False)
print(f'{model_name}算法的best_epoch=',best_epoch)

文件夹models/xgboost已存在
The file 'score_different_algorithms.xlsx' exists.
Index(['MagpieData minimum Number', 'MagpieData maximum Number',
       'MagpieData range Number', 'MagpieData mean Number',
       'MagpieData avg_dev Number', 'MagpieData mode Number',
       'MagpieData minimum MendeleevNumber',
       'MagpieData maximum MendeleevNumber',
       'MagpieData range MendeleevNumber', 'MagpieData mean MendeleevNumber',
       ...
       'avg f valence electrons', 'frac s valence electrons',
       'frac p valence electrons', 'frac d valence electrons',
       'frac f valence electrons', 'Grain Size', 'Habit Plane',
       'Calculated Solid Solution', 'Calculated Grain Boundary',
       'Calculated Yield Strength'],
      dtype='object', length=277)
xgboost_tree_feature_names(top15):
Ca fraction
Interant electrons
MagpieData mean GSvolume_pa
frac d valence electrons
frac f valence electrons
Mixing enthalpy
Yang delta
MagpieData range GSvolume_pa
Zr fraction
MagpieData mean NfValence

In [4]:
import pandas as pd

# 读取Excel文件
df = pd.read_excel('test_set_new.xlsx',index_col = 0)

# 筛选'Habit Plane'和'Precipitate Distribution'均为0的数据
no_precipitate_data = df[(df['Habit Plane'] == 0) |(df['Precipitate Distribution'] == 0)]

# 筛选其他数据
with_precipitate_data = df[(df['Habit Plane'] != 0) & (df['Precipitate Distribution'] != 0)]

# 保存数据到新的Excel文件
no_precipitate_data.to_excel('No_precipitate_data.xlsx', index=False)
with_precipitate_data.to_excel('With_precipitate_data.xlsx', index=False)

# 评估一下不同模型预测性能
import joblib
from sklearn.metrics import mean_squared_error, r2_score

import pandas as pd
import xgboost as xgb
from sklearn.metrics import accuracy_score

def load_data(filepath):
    """加载数据并返回特征和标签"""
    df = pd.read_excel(filepath,)
    X = df.drop(columns=['Precipitate Distribution', '屈服强度', '抗拉强度 (UTS)', '追踪编号'])  # 假设标签列名为'LabelColumn'
    y = df['抗拉强度 (UTS)']
    return X, y

def evaluate_model(model, X, y):
    """使用模型进行预测并评估回归精度"""
    y_pred = model.predict(X)
    mse = mean_squared_error(y, y_pred)
    r2 = r2_score(y, y_pred)
    return mse, r2

# 加载已知的XGBoost模型
model=joblib.load('models/xgboost/xgboost_best_new.pkl')

# 加载两个数据集
X_no_precip, y_no_precip = load_data('No_precipitate_data.xlsx')
X_with_precip, y_with_precip = load_data('With_precipitate_data.xlsx')

# 计算并输出两个数据集的MSE和R^2
mse_no_precip, r2_no_precip = evaluate_model(model, X_no_precip, y_no_precip)
mse_with_precip, r2_with_precip = evaluate_model(model, X_with_precip, y_with_precip)

print(f'MSE on No Precipitate Data: {mse_no_precip}, R^2: {r2_no_precip}')
print(f'MSE on With Precipitate Data: {mse_with_precip}, R^2: {r2_with_precip}')


MSE on No Precipitate Data: 647.4050880772887, R^2: 0.8228190389125827
MSE on With Precipitate Data: 1615.2267427191834, R^2: 0.8294419481620179


In [5]:
# 这里采用随机森林算法进行预测
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold,train_test_split,cross_validate,cross_val_score,KFold
from sklearn.metrics import r2_score, mean_absolute_percentage_error,mean_absolute_error,make_scorer
from sklearn.ensemble import RandomForestRegressor
from utils import create_empty_ls,create_dir,mycopyfile,mycopydir
import joblib
from openpyxl import load_workbook
import os
"""
路径的配置和选择
"""
model_name='rf'

save_model=True
del_0qf=True #是否删除0屈服强度数据
main_path=f'models/{model_name}' # 这里是模型所在的文件位置
create_dir(main_path,is_mainpath=True)
excel=f'index/{model_name}_index.xlsx'
# 如果分数表格不存在就新建一个
score_path='score_different_algorithms.xlsx'
if os.path.exists(score_path):
    print(f"The file '{score_path}' exists.")
else:
    print(f"The file '{score_path}' does not exist.")
    df = pd.DataFrame(columns=['Model', 'R^2', 'MAPE', 'MAE'])
    df.to_excel('score_different_algorithms.xlsx', index=False)


"""
文件读取部分
"""
# 读取 Excel 文件
df = pd.read_excel('train_set_new.xlsx', index_col=0)  # 引入这一列之后，原本的第一列就是一个索引，第0列会从有意义的列开始
# 删除包含空值的行
if del_0qf:
    df = df[df['屈服强度'] != 0].reset_index(drop=True)
np.random.seed(200) #设置随机数种子
feature_names = df.drop(columns=['Precipitate Distribution', '屈服强度', '抗拉强度 (UTS)', '追踪编号']).columns 
print(feature_names)
X = df.drop(columns=['Precipitate Distribution', '屈服强度', '抗拉强度 (UTS)', '追踪编号'])  # 特征: 最后四列之前的所有列
y = df.iloc[:, -2]  # 目标: 倒数第四列
# 将目标变量分箱为10个类别
bins = np.linspace(y.min(), y.max(), 11)
y_binned = np.digitize(y, bins)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

"""
设置初始化
"""
n_splits=5
# 初始化StratifiedKFold
# 初始化XGBoost回归器
# model = XGBRegressor(random_state=200,gamma=0.2)# 设置随机种子以确保结果的可重复性
# 初始化列表以存储每个折叠的分数
# 定义评估指标
scoring = {
    'r2':make_scorer(r2_score,greater_is_better=True),
    'MAE': make_scorer(mean_absolute_error, greater_is_better=False),
    'MAPE': make_scorer(mean_absolute_percentage_error, greater_is_better=False)
}
r2_scores_ls = []
mape_scores = []
mae_scores=[]
best_epoch=0
best_r2_score = float('-inf')
best_mae_score= float('-inf')
best_mape_score= float('-inf')
ls1=['fold'+str(i) for i in range(1,n_splits+1)]
fold_dict=dict(zip(ls1,[0 for i in range(0,len(ls1))]))

num_each_fold_train=np.floor((np.int32(X.shape[0])-1)*(n_splits-1)/n_splits)
num_each_fold_test=(np.int32(X.shape[0])-1)//n_splits

"""
模型训练部分,根据交叉验证结果评估模型性能
"""
epoches=20
for epoch in range(epoches):
    kf = KFold(n_splits=5, shuffle=True, random_state=10*epoch)# 注意这里修改了random_state以确保每次初始化不同
    model =RandomForestRegressor(n_estimators=100, max_depth=40, random_state=200)  # 设置随机种子以确保结果的可重复性
    model.fit(X_train, y_train)
    
    # 计算分折交叉验证结果
    results = cross_validate(model,X,y,cv=kf, scoring=scoring, return_train_score=False)
    # 输出分折交叉验证结果
    for i, (r2, mae, mape) in enumerate(zip(results['test_r2'],results['test_MAE'], results['test_MAPE']), start=1):
        # 记录不同折的分数结果
        r2_scores_ls.append(r2)
        mape_scores.append(-mape)
        mae_scores.append(-mae)
        # print(f"折 {i}: r2={r2:.3f},MAE = {-mae:.3f}, MAPE = {-mape:.3f}")

    current_r2_score=np.mean(r2_scores_ls)
    # scores = cross_val_score(model, X, y_binned, cv=skf, scoring=make_scorer(r2_score))
    # print('r2_scores:',scores)
    # current_r2_score=scores.mean()
    # 只对更优的结果进行保存
    if current_r2_score>best_r2_score:
        # 记录模型的特征重要性
        feature_importance_final=model.feature_importances_
        importance_dict = dict(zip(feature_importance_final, feature_names))
        importance_tuplelist = [(abs(value), feature) for (value, feature) in importance_dict.items()]
        importance_sorted = sorted(importance_tuplelist, reverse=True)
        x_feature = [j for (i, j) in importance_sorted]
        y_importance_value = [i for (i, j) in importance_sorted]
        print(f'{model_name}_tree_feature_names(top15):')
        for i in x_feature[:15]:
            print(i)
        print(f'average_{model_name}_feature_values(top15):')
        for i in y_importance_value[:15]:
            print(i)
        
        #记录更优的分数
        best_epoch=epoch
        best_r2_score=current_r2_score
        best_mae_score=np.mean(mae_scores)
        best_mape_score=np.mean(mape_scores)
        if save_model:
            joblib.dump(model,   f'models/{model_name}/{model_name}_best_new.pkl')
            print(f'保存更优{model_name}模型文件') 
        print('平均 R2 分数:', best_r2_score)
        print('平均 MAPE 分数:', best_mape_score * 100)  # 转换为百分比
        print('平均MAE分数:',best_mae_score)     
"""
打印并输出分数至表格
"""
df = pd.read_excel('score_different_algorithms.xlsx')
new_data = {'Model': f'{model_name}', 'R^2': best_r2_score , 'MAPE': best_mape_score * 100, 'MAE': best_mae_score}
new_data_df = pd.DataFrame([new_data])
# 将新数据添加到数据框中
df = pd.concat([df, new_data_df], ignore_index=True)
# 保存数据框到Excel表格
df.to_excel('score_different_algorithms.xlsx', index=False)
print(f'{model_name}算法的best_epoch=',best_epoch)

文件夹models/rf已存在
The file 'score_different_algorithms.xlsx' exists.
Index(['MagpieData minimum Number', 'MagpieData maximum Number',
       'MagpieData range Number', 'MagpieData mean Number',
       'MagpieData avg_dev Number', 'MagpieData mode Number',
       'MagpieData minimum MendeleevNumber',
       'MagpieData maximum MendeleevNumber',
       'MagpieData range MendeleevNumber', 'MagpieData mean MendeleevNumber',
       ...
       'avg f valence electrons', 'frac s valence electrons',
       'frac p valence electrons', 'frac d valence electrons',
       'frac f valence electrons', 'Grain Size', 'Habit Plane',
       'Calculated Solid Solution', 'Calculated Grain Boundary',
       'Calculated Yield Strength'],
      dtype='object', length=277)
rf_tree_feature_names(top15):
Grain Size
Calculated Yield Strength
Habit Plane
Calculated Grain Boundary
Interant electrons
Interant d electrons
MagpieData mean GSvolume_pa
Zr fraction
Ca fraction
Lambda entropy
MagpieData avg_dev NValence
Ma

In [6]:
# 评估一下不同模型预测性能
import joblib
from sklearn.metrics import mean_squared_error, r2_score

import pandas as pd
import xgboost as xgb
from sklearn.metrics import accuracy_score

def load_data(filepath):
    """加载数据并返回特征和标签"""
    df = pd.read_excel(filepath)
    X = df.drop(columns=['Precipitate Distribution', '屈服强度', '抗拉强度 (UTS)', '追踪编号'])  # 假设标签列名为'LabelColumn'
    y = df['抗拉强度 (UTS)']
    return X, y

def evaluate_model(model, X, y):
    """使用模型进行预测并评估回归精度"""
    y_pred = model.predict(X)
    mse = mean_squared_error(y, y_pred)
    r2 = r2_score(y, y_pred)
    return mse, r2

# 加载已知的XGBoost模型
model=joblib.load('models/rf/rf_best_new.pkl')

# 加载两个数据集
X_no_precip, y_no_precip = load_data('No_precipitate_data.xlsx')
X_with_precip, y_with_precip = load_data('With_precipitate_data.xlsx')

# 计算并输出两个数据集的MSE和R^2
mse_no_precip, r2_no_precip = evaluate_model(model, X_no_precip, y_no_precip)
mse_with_precip, r2_with_precip = evaluate_model(model, X_with_precip, y_with_precip)

print(f'MSE on No Precipitate Data: {mse_no_precip}, R^2: {r2_no_precip}')
print(f'MSE on With Precipitate Data: {mse_with_precip}, R^2: {r2_with_precip}')


MSE on No Precipitate Data: 1228.9434939446621, R^2: 0.6636643835686864
MSE on With Precipitate Data: 1840.5155108446243, R^2: 0.8056528339923459


In [7]:
# 这里采用SVM的回归算法SVR进行预测
from sklearn.svm import SVR
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold,train_test_split,cross_validate,cross_val_score,KFold
from sklearn.metrics import r2_score, mean_absolute_percentage_error,mean_absolute_error,make_scorer
from utils import create_empty_ls,create_dir,mycopyfile,mycopydir
import joblib
from openpyxl import load_workbook
import os
"""
路径的配置和选择
"""
model_name='svm_linear'

save_model=True
del_0qf=True #是否删除0屈服强度数据
main_path=f'models/{model_name}' # 这里是模型所在的文件位置
create_dir(main_path,is_mainpath=True)
# excel=f'index/{model_name}_index.xlsx'
# if os.path.exists(excel):
#     print(f"The file '{excel}' exists.")
# else:
#     print(f"The file '{excel}' does not exist.")

# 如果分数表格不存在就新建一个
score_path='score_different_algorithms.xlsx'
if os.path.exists(score_path):
    print(f"The file '{score_path}' exists.")
else:
    print(f"The file '{score_path}' does not exist.")
    df = pd.DataFrame(columns=['Model', 'R^2', 'MAPE', 'MAE'])
    df.to_excel('score_different_algorithms.xlsx', index=False)


"""
文件读取部分
"""
# 读取 Excel 文件
df = pd.read_excel('train_set_new.xlsx', index_col=0)  # 引入这一列之后，原本的第一列就是一个索引，第0列会从有意义的列开始
# 删除包含空值的行
if del_0qf:
    df = df[df['屈服强度'] != 0].reset_index(drop=True)
np.random.seed(200) #设置随机数种子
feature_names = df.drop(columns=['Precipitate Distribution', '屈服强度', '抗拉强度 (UTS)', '追踪编号']).columns 
print(feature_names)
X = df.drop(columns=['Precipitate Distribution', '屈服强度', '抗拉强度 (UTS)', '追踪编号'])  # 特征: 最后四列之前的所有列
y = df['抗拉强度 (UTS)']  # 目标: 倒数第四列

"""
设置初始化
"""
n_splits=5
# 初始化StratifiedKFold
# 初始化XGBoost回归器
# model = XGBRegressor(random_state=200,gamma=0.2)# 设置随机种子以确保结果的可重复性
# 初始化列表以存储每个折叠的分数
# 定义评估指标
scoring = {
    'r2':make_scorer(r2_score,greater_is_better=True),
    'MAE': make_scorer(mean_absolute_error, greater_is_better=False),
    'MAPE': make_scorer(mean_absolute_percentage_error, greater_is_better=False)
}
r2_scores_ls = []
mape_scores = []
mae_scores=[]
best_epoch=0
best_r2_score = float('-inf')
best_mae_score= float('-inf')
best_mape_score= float('-inf')
ls1=['fold'+str(i) for i in range(1,n_splits+1)]
fold_dict=dict(zip(ls1,[0 for i in range(0,len(ls1))]))

num_each_fold_train=np.floor((np.int32(X.shape[0])-1)*(n_splits-1)/n_splits)
num_each_fold_test=(np.int32(X.shape[0])-1)//n_splits

"""
模型训练部分,根据交叉验证结果评估模型性能
"""
epoches=1
for epoch in range(epoches):
    kf = KFold(n_splits=5, shuffle=True, random_state=60)# 注意这里修改了random_state以确保每次初始化不同
    model = SVR(kernel='linear', C=1.0, epsilon=0.1, gamma='scale')  # 使用SVR模型及SVR的默认参数，
    model.fit(X_train, y_train)
    
    # 计算分折交叉验证结果
    results = cross_validate(model,X,y,cv=kf, scoring=scoring, return_train_score=False)
    # 输出分折交叉验证结果
    for i, (r2, mae, mape) in enumerate(zip(results['test_r2'],results['test_MAE'], results['test_MAPE']), start=1):
        # 记录不同折的分数结果
        r2_scores_ls.append(r2)
        mape_scores.append(-mape)
        mae_scores.append(-mae)
        # print(f"折 {i}: r2={r2:.3f},MAE = {-mae:.3f}, MAPE = {-mape:.3f}")

    current_r2_score=np.mean(r2_scores_ls)
    # scores = cross_val_score(model, X, y_binned, cv=skf, scoring=make_scorer(r2_score))
    # print('r2_scores:',scores)
    # current_r2_score=scores.mean()
    # 只对更优的结果进行保存
    if current_r2_score>best_r2_score:
        # 记录模型的特征重要性
        coefs = model.coef_[0]
        indices = np.argsort(np.abs(coefs))[::-1]
        print(f'{model_name}_linear_kernel_feature_names(top15):')
        for i in feature_names[indices[:15]]:
            print(i)
        print(f'average_{model_name}_feature_values(top15):')
        for i in indices[:15]:
            print(coefs[i])
        
        
        #记录更优的分数
        best_epoch=epoch
        best_r2_score=current_r2_score
        best_mae_score=np.mean(mae_scores)
        best_mape_score=np.mean(mape_scores)
        if save_model:
            joblib.dump(model,   f'models/{model_name}/{model_name}_best_new.pkl')
            print(f'保存更优{model_name}模型文件') 
        print('平均 R2 分数:', best_r2_score)
        print('平均 MAPE 分数:', best_mape_score * 100)  # 转换为百分比
        print('平均MAE分数:',best_mae_score)
        

    
        
"""
打印并输出分数至表格
"""

df = pd.read_excel('score_different_algorithms.xlsx')
new_data = {'Model': f'{model_name}', 'R^2': best_r2_score , 'MAPE': best_mape_score * 100, 'MAE': best_mae_score}
new_data_df = pd.DataFrame([new_data])
# 将新数据添加到数据框中
df = pd.concat([df, new_data_df], ignore_index=True)
# 保存数据框到Excel表格
df.to_excel('score_different_algorithms.xlsx', index=False)
print(f'{model_name}算法的best_epoch=',best_epoch)




文件夹models/svm_linear已存在
The file 'score_different_algorithms.xlsx' exists.
Index(['MagpieData minimum Number', 'MagpieData maximum Number',
       'MagpieData range Number', 'MagpieData mean Number',
       'MagpieData avg_dev Number', 'MagpieData mode Number',
       'MagpieData minimum MendeleevNumber',
       'MagpieData maximum MendeleevNumber',
       'MagpieData range MendeleevNumber', 'MagpieData mean MendeleevNumber',
       ...
       'avg f valence electrons', 'frac s valence electrons',
       'frac p valence electrons', 'frac d valence electrons',
       'frac f valence electrons', 'Grain Size', 'Habit Plane',
       'Calculated Solid Solution', 'Calculated Grain Boundary',
       'Calculated Yield Strength'],
      dtype='object', length=277)
svm_linear_linear_kernel_feature_names(top15):
Interant d electrons
MagpieData avg_dev CovalentRadius
Habit Plane
Interant f electrons
Radii local mismatch
MagpieData mean MendeleevNumber
MagpieData range GSmagmom
MagpieData maximum G

In [8]:
# 使用svm的rbf核进行训练
from sklearn.inspection import permutation_importance
from sklearn.svm import SVR
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold,train_test_split,cross_validate,cross_val_score,KFold
from sklearn.metrics import r2_score, mean_absolute_percentage_error,mean_absolute_error,make_scorer
from utils import create_empty_ls,create_dir,mycopyfile,mycopydir
import joblib
from openpyxl import load_workbook
import os
"""
路径的配置和选择
"""
model_name='svm_rbf'

save_model=True
del_0qf=True #是否删除0屈服强度数据
main_path=f'models/{model_name}' # 这里是模型所在的文件位置
create_dir(main_path,is_mainpath=True)
# excel=f'index/{model_name}_index.xlsx'
# if os.path.exists(excel):
#     print(f"The file '{excel}' exists.")
# else:
#     print(f"The file '{excel}' does not exist.")

# 如果分数表格不存在就新建一个
score_path='score_different_algorithms.xlsx'
if os.path.exists(score_path):
    print(f"The file '{score_path}' exists.")
else:
    print(f"The file '{score_path}' does not exist.")
    df = pd.DataFrame(columns=['Model', 'R^2', 'MAPE', 'MAE'])
    df.to_excel('score_different_algorithms.xlsx', index=False)


"""
文件读取部分
"""
# 读取 Excel 文件
df = pd.read_excel('train_set_new.xlsx', index_col=0)  # 引入这一列之后，原本的第一列就是一个索引，第0列会从有意义的列开始
# 删除包含空值的行
if del_0qf:
    df = df[df['屈服强度'] != 0].reset_index(drop=True)
np.random.seed(200) #设置随机数种子
feature_names = df.drop(columns=['Precipitate Distribution', '屈服强度', '抗拉强度 (UTS)', '追踪编号']).columns 
print(feature_names)
X = df.drop(columns=['Precipitate Distribution', '屈服强度', '抗拉强度 (UTS)', '追踪编号'])  # 特征: 最后四列之前的所有列
y = df['抗拉强度 (UTS)']
"""
设置初始化
"""
n_splits=5
# 初始化StratifiedKFold
# 初始化XGBoost回归器
# model = XGBRegressor(random_state=200,gamma=0.2)# 设置随机种子以确保结果的可重复性
# 初始化列表以存储每个折叠的分数
# 定义评估指标
scoring = {
    'r2':make_scorer(r2_score,greater_is_better=True),
    'MAE': make_scorer(mean_absolute_error, greater_is_better=False),
    'MAPE': make_scorer(mean_absolute_percentage_error, greater_is_better=False)
}
r2_scores_ls = []
mape_scores = []
mae_scores=[]
best_epoch=0
best_r2_score = float('-inf')
best_mae_score= float('-inf')
best_mape_score= float('-inf')
ls1=['fold'+str(i) for i in range(1,n_splits+1)]
fold_dict=dict(zip(ls1,[0 for i in range(0,len(ls1))]))

num_each_fold_train=np.floor((np.int32(X.shape[0])-1)*(n_splits-1)/n_splits)
num_each_fold_test=(np.int32(X.shape[0])-1)//n_splits

"""
模型训练部分,根据交叉验证结果评估模型性能
"""
epoches=1
for epoch in range(epoches):
    kf = KFold(n_splits=5, shuffle=True, random_state=60)# 注意这里修改了random_state以确保每次初始化不同
    model = SVR(kernel='rbf', C=1.0, epsilon=0.2, gamma='scale')  # 使用SVR模型及SVR的默认参数，
    model.fit(X_train, y_train)
    
    # 计算分折交叉验证结果
    results = cross_validate(model,X,y,cv=kf, scoring=scoring, return_train_score=False)
    # 输出分折交叉验证结果
    for i, (r2, mae, mape) in enumerate(zip(results['test_r2'],results['test_MAE'], results['test_MAPE']), start=1):
        # 记录不同折的分数结果
        r2_scores_ls.append(r2)
        mape_scores.append(-mape)
        mae_scores.append(-mae)
        # print(f"折 {i}: r2={r2:.3f},MAE = {-mae:.3f}, MAPE = {-mape:.3f}")

    current_r2_score=np.mean(r2_scores_ls)
    # scores = cross_val_score(model, X, y_binned, cv=skf, scoring=make_scorer(r2_score))
    # print('r2_scores:',scores)
    # current_r2_score=scores.mean()
    # 只对更优的结果进行保存
    if current_r2_score>best_r2_score:
        # 记录模型的特征重要性
        perm_importance = permutation_importance(model, X_test, y_test, n_repeats=30, random_state=42)
        # 获取特征重要性值
        importances = perm_importance.importances_mean

        # 对特征重要性进行排序
        sorted_idx = np.argsort(importances)[::-1]
        
        print(f'{model_name}_feature_names(top15):')
        for i in feature_names[sorted_idx[:15]]:
            print(i)
        print(f'average_{model_name}_feature_values(top15):')
        for i in importances[sorted_idx[:15]]:
            print(i)
        
        
        #记录更优的分数
        best_epoch=epoch
        best_r2_score=current_r2_score
        best_mae_score=np.mean(mae_scores)
        best_mape_score=np.mean(mape_scores)
        if save_model:
            joblib.dump(model,   f'models/{model_name}/{model_name}_best_new.pkl')
            print(f'保存更优{model_name}模型文件') 
        print('平均 R2 分数:', best_r2_score)
        print('平均 MAPE 分数:', best_mape_score * 100)  # 转换为百分比
        print('平均MAE分数:',best_mae_score)
        

    
        
"""
打印并输出分数至表格
"""

df = pd.read_excel('score_different_algorithms.xlsx')
new_data = {'Model': f'{model_name}', 'R^2': best_r2_score , 'MAPE': best_mape_score * 100, 'MAE': best_mae_score}
new_data_df = pd.DataFrame([new_data])
# 将新数据添加到数据框中
df = pd.concat([df, new_data_df], ignore_index=True)
# 保存数据框到Excel表格
df.to_excel('score_different_algorithms.xlsx', index=False)
print(f'{model_name}算法的best_epoch=',best_epoch)

文件夹models/svm_rbf已存在
The file 'score_different_algorithms.xlsx' exists.
Index(['MagpieData minimum Number', 'MagpieData maximum Number',
       'MagpieData range Number', 'MagpieData mean Number',
       'MagpieData avg_dev Number', 'MagpieData mode Number',
       'MagpieData minimum MendeleevNumber',
       'MagpieData maximum MendeleevNumber',
       'MagpieData range MendeleevNumber', 'MagpieData mean MendeleevNumber',
       ...
       'avg f valence electrons', 'frac s valence electrons',
       'frac p valence electrons', 'frac d valence electrons',
       'frac f valence electrons', 'Grain Size', 'Habit Plane',
       'Calculated Solid Solution', 'Calculated Grain Boundary',
       'Calculated Yield Strength'],
      dtype='object', length=277)
svm_rbf_feature_names(top15):
Calculated Yield Strength
Calculated Grain Boundary
Grain Size
MagpieData maximum AtomicWeight
MagpieData range AtomicWeight
Calculated Solid Solution
MagpieData maximum Number
MagpieData range Number
range 

In [9]:
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold,train_test_split,cross_validate,cross_val_score,KFold
from sklearn.metrics import r2_score, mean_absolute_percentage_error,mean_absolute_error,make_scorer
from utils import create_empty_ls,create_dir,mycopyfile,mycopydir
import joblib
from openpyxl import load_workbook
import os
"""
路径的配置和选择
"""
model_name='ridge'

save_model=True
del_0qf=True #是否删除0屈服强度数据
main_path=f'models/{model_name}' # 这里是模型所在的文件位置
create_dir(main_path,is_mainpath=True)
# excel=f'index/{model_name}_index.xlsx'
# if os.path.exists(excel):
#     print(f"The file '{excel}' exists.")
# else:
#     print(f"The file '{excel}' does not exist.")

# 如果分数表格不存在就新建一个
score_path='score_different_algorithms.xlsx'
if os.path.exists(score_path):
    print(f"The file '{score_path}' exists.")
else:
    print(f"The file '{score_path}' does not exist.")
    df = pd.DataFrame(columns=['Model', 'R^2', 'MAPE', 'MAE'])
    df.to_excel('score_different_algorithms.xlsx', index=False)

"""
文件读取部分
"""
# 读取 Excel 文件
df = pd.read_excel('train_set_new.xlsx', index_col=0)  # 引入这一列之后，原本的第一列就是一个索引，第0列会从有意义的列开始
# 删除包含空值的行
if del_0qf:
    df = df[df['屈服强度'] != 0].reset_index(drop=True)
np.random.seed(200) #设置随机数种子
feature_names = df.drop(columns=['Precipitate Distribution', '屈服强度', '抗拉强度 (UTS)', '追踪编号']).columns 
print(feature_names)
X = df.drop(columns=['Precipitate Distribution', '屈服强度', '抗拉强度 (UTS)', '追踪编号'])  # 特征: 最后四列之前的所有列
y = df['抗拉强度 (UTS)']
"""
设置初始化
"""
n_splits=5
# 初始化StratifiedKFold
# 初始化XGBoost回归器
# model = XGBRegressor(random_state=200,gamma=0.2)# 设置随机种子以确保结果的可重复性
# 初始化列表以存储每个折叠的分数
# 定义评估指标
scoring = {
    'r2':make_scorer(r2_score,greater_is_better=True),
    'MAE': make_scorer(mean_absolute_error, greater_is_better=False),
    'MAPE': make_scorer(mean_absolute_percentage_error, greater_is_better=False)
}
r2_scores_ls = []
mape_scores = []
mae_scores=[]
best_epoch=0
best_r2_score = float('-inf')
best_mae_score= float('-inf')
best_mape_score= float('-inf')
ls1=['fold'+str(i) for i in range(1,n_splits+1)]
fold_dict=dict(zip(ls1,[0 for i in range(0,len(ls1))]))

num_each_fold_train=np.floor((np.int32(X.shape[0])-1)*(n_splits-1)/n_splits)
num_each_fold_test=(np.int32(X.shape[0])-1)//n_splits

"""
模型训练部分,根据交叉验证结果评估模型性能
"""
epoches=1
for epoch in range(epoches):
    kf = KFold(n_splits=5, shuffle=True, random_state=60)# 注意这里修改了random_state以确保每次初始化不同
    model = Ridge(alpha=1.0) 
    model.fit(X_train, y_train)
    
    # 计算分折交叉验证结果
    results = cross_validate(model,X,y,cv=kf, scoring=scoring, return_train_score=False)
    # 输出分折交叉验证结果
    for i, (r2, mae, mape) in enumerate(zip(results['test_r2'],results['test_MAE'], results['test_MAPE']), start=1):
        # 记录不同折的分数结果
        r2_scores_ls.append(r2)
        mape_scores.append(-mape)
        mae_scores.append(-mae)
        # print(f"折 {i}: r2={r2:.3f},MAE = {-mae:.3f}, MAPE = {-mape:.3f}")

    current_r2_score=np.mean(r2_scores_ls)
    # scores = cross_val_score(model, X, y_binned, cv=skf, scoring=make_scorer(r2_score))
    # print('r2_scores:',scores)
    # current_r2_score=scores.mean()
    # 只对更优的结果进行保存
    if current_r2_score>best_r2_score:
        # 记录模型的特征重要性
        perm_importance = permutation_importance(model, X_test, y_test, n_repeats=30, random_state=42)
        # 获取特征重要性值
        importances = perm_importance.importances_mean

        # 对特征重要性进行排序
        sorted_idx = np.argsort(importances)[::-1]
        
        print(f'{model_name}_feature_names(top15):')
        for i in feature_names[sorted_idx[:15]]:
            print(i)
        print(f'average_{model_name}_feature_values(top15):')
        for i in importances[sorted_idx[:15]]:
            print(i)
        
        
        #记录更优的分数
        best_epoch=epoch
        best_r2_score=current_r2_score
        best_mae_score=np.mean(mae_scores)
        best_mape_score=np.mean(mape_scores)
        if save_model:
            joblib.dump(model,   f'models/{model_name}/{model_name}_best_new.pkl')
            print(f'保存更优{model_name}模型文件') 
        print('平均 R2 分数:', best_r2_score)
        print('平均 MAPE 分数:', best_mape_score * 100)  # 转换为百分比
        print('平均MAE分数:',best_mae_score)
        

    
        
"""
打印并输出分数至表格
"""

df = pd.read_excel('score_different_algorithms.xlsx')
new_data = {'Model': f'{model_name}', 'R^2': best_r2_score , 'MAPE': best_mape_score * 100, 'MAE': best_mae_score}
new_data_df = pd.DataFrame([new_data])
# 将新数据添加到数据框中
df = pd.concat([df, new_data_df], ignore_index=True)
# 保存数据框到Excel表格
df.to_excel('score_different_algorithms.xlsx', index=False)
print(f'{model_name}算法的best_epoch=',best_epoch)

文件夹models/ridge已存在
The file 'score_different_algorithms.xlsx' exists.
Index(['MagpieData minimum Number', 'MagpieData maximum Number',
       'MagpieData range Number', 'MagpieData mean Number',
       'MagpieData avg_dev Number', 'MagpieData mode Number',
       'MagpieData minimum MendeleevNumber',
       'MagpieData maximum MendeleevNumber',
       'MagpieData range MendeleevNumber', 'MagpieData mean MendeleevNumber',
       ...
       'avg f valence electrons', 'frac s valence electrons',
       'frac p valence electrons', 'frac d valence electrons',
       'frac f valence electrons', 'Grain Size', 'Habit Plane',
       'Calculated Solid Solution', 'Calculated Grain Boundary',
       'Calculated Yield Strength'],
      dtype='object', length=277)
ridge_feature_names(top15):
MagpieData range MendeleevNumber
MagpieData minimum MendeleevNumber
Calculated Yield Strength
range Number
MagpieData range Number
MagpieData maximum MendeleevNumber
MagpieData mean MendeleevNumber
MagpieData ma

In [10]:
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Lasso 
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold,train_test_split,cross_validate,cross_val_score,KFold
from sklearn.metrics import r2_score, mean_absolute_percentage_error,mean_absolute_error,make_scorer
from utils import create_empty_ls,create_dir,mycopyfile,mycopydir
import joblib
from openpyxl import load_workbook
import os
"""
路径的配置和选择
"""
model_name='lasso'

save_model=True
del_0qf=True #是否删除0屈服强度数据
main_path=f'models/{model_name}' # 这里是模型所在的文件位置
create_dir(main_path,is_mainpath=True)
# excel=f'index/{model_name}_index.xlsx'
# if os.path.exists(excel):
#     print(f"The file '{excel}' exists.")
# else:
#     print(f"The file '{excel}' does not exist.")

# 如果分数表格不存在就新建一个
score_path='score_different_algorithms.xlsx'
if os.path.exists(score_path):
    print(f"The file '{score_path}' exists.")
else:
    print(f"The file '{score_path}' does not exist.")
    df = pd.DataFrame(columns=['Model', 'R^2', 'MAPE', 'MAE'])
    df.to_excel('score_different_algorithms.xlsx', index=False)


"""
文件读取部分
"""
# 读取 Excel 文件
df = pd.read_excel('train_set_new.xlsx', index_col=0)  # 引入这一列之后，原本的第一列就是一个索引，第0列会从有意义的列开始
# 删除包含空值的行
if del_0qf:
    df = df[df['屈服强度'] != 0].reset_index(drop=True)
np.random.seed(200) #设置随机数种子
feature_names = df.drop(columns=['Precipitate Distribution', '屈服强度', '抗拉强度 (UTS)', '追踪编号']).columns 
print(feature_names)
X = df.drop(columns=['Precipitate Distribution', '屈服强度', '抗拉强度 (UTS)', '追踪编号'])  # 特征: 最后四列之前的所有列
y = df['抗拉强度 (UTS)']
"""
设置初始化
"""
n_splits=5
# 初始化StratifiedKFold
# 初始化XGBoost回归器
# model = XGBRegressor(random_state=200,gamma=0.2)# 设置随机种子以确保结果的可重复性
# 初始化列表以存储每个折叠的分数
# 定义评估指标
scoring = {
    'r2':make_scorer(r2_score,greater_is_better=True),
    'MAE': make_scorer(mean_absolute_error, greater_is_better=False),
    'MAPE': make_scorer(mean_absolute_percentage_error, greater_is_better=False)
}
r2_scores_ls = []
mape_scores = []
mae_scores=[]
best_epoch=0
best_r2_score = float('-inf')
best_mae_score= float('-inf')
best_mape_score= float('-inf')
ls1=['fold'+str(i) for i in range(1,n_splits+1)]
fold_dict=dict(zip(ls1,[0 for i in range(0,len(ls1))]))

num_each_fold_train=np.floor((np.int32(X.shape[0])-1)*(n_splits-1)/n_splits)
num_each_fold_test=(np.int32(X.shape[0])-1)//n_splits

"""
模型训练部分,根据交叉验证结果评估模型性能
"""
epoches=1
for epoch in range(epoches):
    kf = KFold(n_splits=5, shuffle=True, random_state=60)# 注意这里修改了random_state以确保每次初始化不同
    model = Lasso(alpha=1.0)
    model.fit(X_train, y_train)
    
    # 计算分折交叉验证结果
    results = cross_validate(model,X,y,cv=kf, scoring=scoring, return_train_score=False)
    # 输出分折交叉验证结果
    for i, (r2, mae, mape) in enumerate(zip(results['test_r2'],results['test_MAE'], results['test_MAPE']), start=1):
        # 记录不同折的分数结果
        r2_scores_ls.append(r2)
        mape_scores.append(-mape)
        mae_scores.append(-mae)
        # print(f"折 {i}: r2={r2:.3f},MAE = {-mae:.3f}, MAPE = {-mape:.3f}")

    current_r2_score=np.mean(r2_scores_ls)
    # scores = cross_val_score(model, X, y_binned, cv=skf, scoring=make_scorer(r2_score))
    # print('r2_scores:',scores)
    # current_r2_score=scores.mean()
    # 只对更优的结果进行保存
    if current_r2_score>best_r2_score:
        # 记录模型的特征重要性
        perm_importance = permutation_importance(model, X_test, y_test, n_repeats=30, random_state=42)
        # 获取特征重要性值
        importances = perm_importance.importances_mean

        # 对特征重要性进行排序
        sorted_idx = np.argsort(importances)[::-1]
        
        print(f'{model_name}_feature_names(top15):')
        for i in feature_names[sorted_idx[:15]]:
            print(i)
        print(f'average_{model_name}_feature_values(top15):')
        for i in importances[sorted_idx[:15]]:
            print(i)
        #记录更优的分数
        best_epoch=epoch
        best_r2_score=current_r2_score
        best_mae_score=np.mean(mae_scores)
        best_mape_score=np.mean(mape_scores)
        if save_model:
            joblib.dump(model,   f'models/{model_name}/{model_name}_best_new.pkl')
            print(f'保存更优{model_name}模型文件') 
        print('平均 R2 分数:', best_r2_score)
        print('平均 MAPE 分数:', best_mape_score * 100)  # 转换为百分比
        print('平均MAE分数:',best_mae_score)
       
"""
打印并输出分数至表格
"""

df = pd.read_excel('score_different_algorithms.xlsx')
new_data = {'Model': f'{model_name}', 'R^2': best_r2_score , 'MAPE': best_mape_score * 100, 'MAE': best_mae_score}
new_data_df = pd.DataFrame([new_data])
# 将新数据添加到数据框中
df = pd.concat([df, new_data_df], ignore_index=True)
# 保存数据框到Excel表格
df.to_excel('score_different_algorithms.xlsx', index=False)
print(f'{model_name}算法的best_epoch=',best_epoch)


文件夹models/lasso已存在
The file 'score_different_algorithms.xlsx' exists.
Index(['MagpieData minimum Number', 'MagpieData maximum Number',
       'MagpieData range Number', 'MagpieData mean Number',
       'MagpieData avg_dev Number', 'MagpieData mode Number',
       'MagpieData minimum MendeleevNumber',
       'MagpieData maximum MendeleevNumber',
       'MagpieData range MendeleevNumber', 'MagpieData mean MendeleevNumber',
       ...
       'avg f valence electrons', 'frac s valence electrons',
       'frac p valence electrons', 'frac d valence electrons',
       'frac f valence electrons', 'Grain Size', 'Habit Plane',
       'Calculated Solid Solution', 'Calculated Grain Boundary',
       'Calculated Yield Strength'],
      dtype='object', length=277)
lasso_feature_names(top15):
range Number
Calculated Solid Solution
MagpieData range MendeleevNumber
MagpieData minimum GSvolume_pa
MagpieData maximum NdUnfilled
MagpieData mean AtomicWeight
Radii local mismatch
Interant d electrons
Calcula

In [11]:
from sklearn.linear_model import ElasticNet  # 导入ElasticNet
from sklearn.inspection import permutation_importance
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold,train_test_split,cross_validate,cross_val_score,KFold
from sklearn.metrics import r2_score, mean_absolute_percentage_error,mean_absolute_error,make_scorer
from utils import create_empty_ls,create_dir,mycopyfile,mycopydir
import joblib
from openpyxl import load_workbook
import os
"""
路径的配置和选择
"""
model_name='elasticNet'

save_model=True
del_0qf=True #是否删除0屈服强度数据
main_path=f'models/{model_name}' # 这里是模型所在的文件位置
create_dir(main_path,is_mainpath=True)
# excel=f'index/{model_name}_index.xlsx'
# if os.path.exists(excel):
#     print(f"The file '{excel}' exists.")
# else:
#     print(f"The file '{excel}' does not exist.")

# 如果分数表格不存在就新建一个
score_path='score_different_algorithms.xlsx'
if os.path.exists(score_path):
    print(f"The file '{score_path}' exists.")
else:
    print(f"The file '{score_path}' does not exist.")
    df = pd.DataFrame(columns=['Model', 'R^2', 'MAPE', 'MAE'])
    df.to_excel('score_different_algorithms.xlsx', index=False)


"""
文件读取部分
"""
# 读取 Excel 文件
df = pd.read_excel('train_set_new.xlsx', index_col=0)  # 引入这一列之后，原本的第一列就是一个索引，第0列会从有意义的列开始
# 删除包含空值的行
if del_0qf:
    df = df[df['屈服强度'] != 0].reset_index(drop=True)
np.random.seed(200) #设置随机数种子
feature_names = df.drop(columns=['Precipitate Distribution', '屈服强度', '抗拉强度 (UTS)', '追踪编号']).columns 
print(feature_names)
X = df.drop(columns=['Precipitate Distribution', '屈服强度', '抗拉强度 (UTS)', '追踪编号'])  # 特征: 最后四列之前的所有列
y = df['抗拉强度 (UTS)']
"""
设置初始化
"""
n_splits=5
# 初始化StratifiedKFold
# 初始化XGBoost回归器
# model = XGBRegressor(random_state=200,gamma=0.2)# 设置随机种子以确保结果的可重复性
# 初始化列表以存储每个折叠的分数
# 定义评估指标
scoring = {
    'r2':make_scorer(r2_score,greater_is_better=True),
    'MAE': make_scorer(mean_absolute_error, greater_is_better=False),
    'MAPE': make_scorer(mean_absolute_percentage_error, greater_is_better=False)
}
r2_scores_ls = []
mape_scores = []
mae_scores=[]
best_epoch=0
best_r2_score = float('-inf')
best_mae_score= float('-inf')
best_mape_score= float('-inf')
ls1=['fold'+str(i) for i in range(1,n_splits+1)]
fold_dict=dict(zip(ls1,[0 for i in range(0,len(ls1))]))

num_each_fold_train=np.floor((np.int32(X.shape[0])-1)*(n_splits-1)/n_splits)
num_each_fold_test=(np.int32(X.shape[0])-1)//n_splits

"""
模型训练部分,根据交叉验证结果评估模型性能
"""
epoches=1
for epoch in range(epoches):
    kf = KFold(n_splits=5, shuffle=True, random_state=60)# 注意这里修改了random_state以确保每次初始化不同
    model = ElasticNet(alpha=1.0, l1_ratio=0.5)  # 初始化ElasticNet回归模型，alpha是正则化强度，l1_ratio决定正则化类型的混合比例
    model.fit(X_train, y_train)
    
    # 计算分折交叉验证结果
    results = cross_validate(model,X,y,cv=kf, scoring=scoring, return_train_score=False)
    # 输出分折交叉验证结果
    for i, (r2, mae, mape) in enumerate(zip(results['test_r2'],results['test_MAE'], results['test_MAPE']), start=1):
        # 记录不同折的分数结果
        r2_scores_ls.append(r2)
        mape_scores.append(-mape)
        mae_scores.append(-mae)
        # print(f"折 {i}: r2={r2:.3f},MAE = {-mae:.3f}, MAPE = {-mape:.3f}")

    current_r2_score=np.mean(r2_scores_ls)
    # scores = cross_val_score(model, X, y_binned, cv=skf, scoring=make_scorer(r2_score))
    # print('r2_scores:',scores)
    # current_r2_score=scores.mean()
    # 只对更优的结果进行保存
    if current_r2_score>best_r2_score:
        # 记录模型的特征重要性
        perm_importance = permutation_importance(model, X_test, y_test, n_repeats=30, random_state=42)
        # 获取特征重要性值
        importances = perm_importance.importances_mean

        # 对特征重要性进行排序
        sorted_idx = np.argsort(importances)[::-1]
        
        print(f'{model_name}_feature_names(top15):')
        for i in feature_names[sorted_idx[:15]]:
            print(i)
        print(f'average_{model_name}_feature_values(top15):')
        for i in importances[sorted_idx[:15]]:
            print(i)
        
        
        #记录更优的分数
        best_epoch=epoch
        best_r2_score=current_r2_score
        best_mae_score=np.mean(mae_scores)
        best_mape_score=np.mean(mape_scores)
        if save_model:
            joblib.dump(model,   f'models/{model_name}/{model_name}_best_new.pkl')
            print(f'保存更优{model_name}模型文件') 
        print('平均 R2 分数:', best_r2_score)
        print('平均 MAPE 分数:', best_mape_score * 100)  # 转换为百分比
        print('平均MAE分数:',best_mae_score)
        

    
        
"""
打印并输出分数至表格
"""

df = pd.read_excel('score_different_algorithms.xlsx')
new_data = {'Model': f'{model_name}', 'R^2': best_r2_score , 'MAPE': best_mape_score * 100, 'MAE': best_mae_score}
new_data_df = pd.DataFrame([new_data])
# 将新数据添加到数据框中
df = pd.concat([df, new_data_df], ignore_index=True)
# 保存数据框到Excel表格
df.to_excel('score_different_algorithms.xlsx', index=False)
print(f'{model_name}算法的best_epoch=',best_epoch)




文件夹models/elasticNet已存在
The file 'score_different_algorithms.xlsx' exists.
Index(['MagpieData minimum Number', 'MagpieData maximum Number',
       'MagpieData range Number', 'MagpieData mean Number',
       'MagpieData avg_dev Number', 'MagpieData mode Number',
       'MagpieData minimum MendeleevNumber',
       'MagpieData maximum MendeleevNumber',
       'MagpieData range MendeleevNumber', 'MagpieData mean MendeleevNumber',
       ...
       'avg f valence electrons', 'frac s valence electrons',
       'frac p valence electrons', 'frac d valence electrons',
       'frac f valence electrons', 'Grain Size', 'Habit Plane',
       'Calculated Solid Solution', 'Calculated Grain Boundary',
       'Calculated Yield Strength'],
      dtype='object', length=277)
elasticNet_feature_names(top15):
MagpieData range MendeleevNumber
range Number
Calculated Solid Solution
MagpieData range Number
MagpieData range AtomicWeight
MagpieData minimum GSvolume_pa
Habit Plane
Calculated Yield Strength
Calcul

In [12]:
from sklearn.tree import DecisionTreeRegressor  # 导入决策树回归器
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold,train_test_split,cross_validate,cross_val_score,KFold
from sklearn.metrics import r2_score, mean_absolute_percentage_error,mean_absolute_error,make_scorer
from utils import create_empty_ls,create_dir,mycopyfile,mycopydir
import joblib
from openpyxl import load_workbook
import os
"""
路径的配置和选择
"""
model_name='dt'

save_model=True
del_0qf=True #是否删除0屈服强度数据
main_path=f'models/{model_name}' # 这里是模型所在的文件位置
create_dir(main_path,is_mainpath=True)
excel=f'index/{model_name}_index.xlsx'
# 如果分数表格不存在就新建一个
score_path='score_different_algorithms.xlsx'
if os.path.exists(score_path):
    print(f"The file '{score_path}' exists.")
else:
    print(f"The file '{score_path}' does not exist.")
    df = pd.DataFrame(columns=['Model', 'R^2', 'MAPE', 'MAE'])
    df.to_excel('score_different_algorithms.xlsx', index=False)


"""
文件读取部分
"""
# 读取 Excel 文件
df = pd.read_excel('train_set_new.xlsx', index_col=0)  # 引入这一列之后，原本的第一列就是一个索引，第0列会从有意义的列开始
# 删除包含空值的行
if del_0qf:
    df = df[df['屈服强度'] != 0].reset_index(drop=True)
np.random.seed(200) #设置随机数种子
feature_names = df.drop(columns=['Precipitate Distribution', '屈服强度', '抗拉强度 (UTS)', '追踪编号']).columns 
print(feature_names)
X = df.drop(columns=['Precipitate Distribution', '屈服强度', '抗拉强度 (UTS)', '追踪编号'])  # 特征: 最后四列之前的所有列
y = df['抗拉强度 (UTS)']
"""
设置初始化
"""
n_splits=5
# 初始化StratifiedKFold
# 初始化XGBoost回归器
# model = XGBRegressor(random_state=200,gamma=0.2)# 设置随机种子以确保结果的可重复性
# 初始化列表以存储每个折叠的分数
# 定义评估指标
scoring = {
    'r2':make_scorer(r2_score,greater_is_better=True),
    'MAE': make_scorer(mean_absolute_error, greater_is_better=False),
    'MAPE': make_scorer(mean_absolute_percentage_error, greater_is_better=False)
}
r2_scores_ls = []
mape_scores = []
mae_scores=[]
best_epoch=0
best_r2_score = float('-inf')
best_mae_score= float('-inf')
best_mape_score= float('-inf')
ls1=['fold'+str(i) for i in range(1,n_splits+1)]
fold_dict=dict(zip(ls1,[0 for i in range(0,len(ls1))]))

num_each_fold_train=np.floor((np.int32(X.shape[0])-1)*(n_splits-1)/n_splits)
num_each_fold_test=(np.int32(X.shape[0])-1)//n_splits

"""
模型训练部分,根据交叉验证结果评估模型性能
"""
epoches=1
for epoch in range(epoches):
    kf = KFold(n_splits=5, shuffle=True, random_state=60)# 注意这里修改了random_state以确保每次初始化不同
    model = DecisionTreeRegressor(random_state=200)
    model.fit(X_train, y_train)

    # 计算分折交叉验证结果
    results = cross_validate(model,X,y,cv=kf, scoring=scoring, return_train_score=False)
    # 输出分折交叉验证结果
    for i, (r2, mae, mape) in enumerate(zip(results['test_r2'],results['test_MAE'], results['test_MAPE']), start=1):
        # 记录不同折的分数结果
        r2_scores_ls.append(r2)
        mape_scores.append(-mape)
        mae_scores.append(-mae)
        # print(f"折 {i}: r2={r2:.3f},MAE = {-mae:.3f}, MAPE = {-mape:.3f}")

    current_r2_score=np.mean(r2_scores_ls)
    # scores = cross_val_score(model, X, y_binned, cv=skf, scoring=make_scorer(r2_score))
    # print('r2_scores:',scores)
    # current_r2_score=scores.mean()
    # 只对更优的结果进行保存
    if current_r2_score>best_r2_score:
        # 记录模型的特征重要性
        
        feature_importance_final=model.feature_importances_
        importance_dict = dict(zip(feature_importance_final, feature_names))
        importance_tuplelist = [(abs(value), feature) for (value, feature) in importance_dict.items()]
        importance_sorted = sorted(importance_tuplelist, reverse=True)
        x_feature = [j for (i, j) in importance_sorted]
        y_importance_value = [i for (i, j) in importance_sorted]
        print(f'{model_name}_tree_feature_names(top15):')
        for i in x_feature[:15]:
            print(i)
        print(f'average_{model_name}_feature_values(top15):')
        for i in y_importance_value[:15]:
            print(i)
        
        #记录更优的分数
        best_epoch=epoch
        best_r2_score=current_r2_score
        best_mae_score=np.mean(mae_scores)
        best_mape_score=np.mean(mape_scores)
        if save_model:
            joblib.dump(model,   f'models/{model_name}/{model_name}_best_new.pkl')
            print(f'保存更优{model_name}模型文件') 
        print('平均 R2 分数:', best_r2_score)
        print('平均 MAPE 分数:', best_mape_score * 100)  # 转换为百分比
        print('平均MAE分数:',best_mae_score)
        

    
        
"""
打印并输出分数至表格
"""

df = pd.read_excel('score_different_algorithms.xlsx')
new_data = {'Model': f'{model_name}', 'R^2': best_r2_score , 'MAPE': best_mape_score * 100, 'MAE': best_mae_score}
new_data_df = pd.DataFrame([new_data])
# 将新数据添加到数据框中
df = pd.concat([df, new_data_df], ignore_index=True)
# 保存数据框到Excel表格
df.to_excel('score_different_algorithms.xlsx', index=False)
print(f'{model_name}算法的best_epoch=',best_epoch)


文件夹models/dt已存在
The file 'score_different_algorithms.xlsx' exists.
Index(['MagpieData minimum Number', 'MagpieData maximum Number',
       'MagpieData range Number', 'MagpieData mean Number',
       'MagpieData avg_dev Number', 'MagpieData mode Number',
       'MagpieData minimum MendeleevNumber',
       'MagpieData maximum MendeleevNumber',
       'MagpieData range MendeleevNumber', 'MagpieData mean MendeleevNumber',
       ...
       'avg f valence electrons', 'frac s valence electrons',
       'frac p valence electrons', 'frac d valence electrons',
       'frac f valence electrons', 'Grain Size', 'Habit Plane',
       'Calculated Solid Solution', 'Calculated Grain Boundary',
       'Calculated Yield Strength'],
      dtype='object', length=277)
dt_tree_feature_names(top15):
Grain Size
Interant electrons
MagpieData mean GSvolume_pa
Calculated Grain Boundary
Zr fraction
Calculated Yield Strength
Habit Plane
Ca fraction
Gd fraction
MagpieData minimum MendeleevNumber
Zn fraction
Mean co

In [13]:
from sklearn.ensemble import GradientBoostingRegressor  # 导入GradientBoostingRegressor
from sklearn.inspection import permutation_importance
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold,train_test_split,cross_validate,cross_val_score,KFold
from sklearn.metrics import r2_score, mean_absolute_percentage_error,mean_absolute_error,make_scorer
from utils import create_empty_ls,create_dir,mycopyfile,mycopydir
import joblib
from openpyxl import load_workbook
import os
"""
路径的配置和选择
"""
model_name='gboost'

save_model=True
del_0qf=True #是否删除0屈服强度数据
main_path=f'models/{model_name}' # 这里是模型所在的文件位置
create_dir(main_path,is_mainpath=True)
# excel=f'index/{model_name}_index.xlsx'
# if os.path.exists(excel):
#     print(f"The file '{excel}' exists.")
# else:
#     print(f"The file '{excel}' does not exist.")

# 如果分数表格不存在就新建一个
score_path='score_different_algorithms.xlsx'
if os.path.exists(score_path):
    print(f"The file '{score_path}' exists.")
else:
    print(f"The file '{score_path}' does not exist.")
    df = pd.DataFrame(columns=['Model', 'R^2', 'MAPE', 'MAE'])
    df.to_excel('score_different_algorithms.xlsx', index=False)


"""
文件读取部分
"""
# 读取 Excel 文件
df = pd.read_excel('train_set_new.xlsx', index_col=0)  # 引入这一列之后，原本的第一列就是一个索引，第0列会从有意义的列开始
# 删除包含空值的行
if del_0qf:
    df = df[df['屈服强度'] != 0].reset_index(drop=True)
np.random.seed(200) #设置随机数种子
feature_names = df.drop(columns=['Precipitate Distribution', '屈服强度', '抗拉强度 (UTS)', '追踪编号']).columns 
print(feature_names)
X = df.drop(columns=['Precipitate Distribution', '屈服强度', '抗拉强度 (UTS)', '追踪编号'])  # 特征: 最后四列之前的所有列
y = df['抗拉强度 (UTS)']
"""
设置初始化
"""
n_splits=5
# 初始化StratifiedKFold
# 初始化XGBoost回归器
# model = XGBRegressor(random_state=200,gamma=0.2)# 设置随机种子以确保结果的可重复性
# 初始化列表以存储每个折叠的分数
# 定义评估指标
scoring = {
    'r2':make_scorer(r2_score,greater_is_better=True),
    'MAE': make_scorer(mean_absolute_error, greater_is_better=False),
    'MAPE': make_scorer(mean_absolute_percentage_error, greater_is_better=False)
}
r2_scores_ls = []
mape_scores = []
mae_scores=[]
best_epoch=0
best_r2_score = float('-inf')
best_mae_score= float('-inf')
best_mape_score= float('-inf')
ls1=['fold'+str(i) for i in range(1,n_splits+1)]
fold_dict=dict(zip(ls1,[0 for i in range(0,len(ls1))]))

num_each_fold_train=np.floor((np.int32(X.shape[0])-1)*(n_splits-1)/n_splits)
num_each_fold_test=(np.int32(X.shape[0])-1)//n_splits

"""
模型训练部分,根据交叉验证结果评估模型性能
"""
epoches=1
for epoch in range(epoches):
    kf = KFold(n_splits=5, shuffle=True, random_state=0+10*epoch)# 设置随机种子以确保结果的可重复性
    model = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=200)
    model.fit(X_train, y_train)
    
    # 计算分折交叉验证结果
    results = cross_validate(model,X,y,cv=kf, scoring=scoring, return_train_score=False)
    # 输出分折交叉验证结果
    for i, (r2, mae, mape) in enumerate(zip(results['test_r2'],results['test_MAE'], results['test_MAPE']), start=1):
        # 记录不同折的分数结果
        r2_scores_ls.append(r2)
        mape_scores.append(-mape)
        mae_scores.append(-mae)
        # print(f"折 {i}: r2={r2:.3f},MAE = {-mae:.3f}, MAPE = {-mape:.3f}")

    current_r2_score=np.mean(r2_scores_ls)
    # scores = cross_val_score(model, X, y_binned, cv=skf, scoring=make_scorer(r2_score))
    # print('r2_scores:',scores)
    # current_r2_score=scores.mean()
    # 只对更优的结果进行保存
    if current_r2_score>best_r2_score:
        best_epoch=epoch
        # 记录模型的特征重要性
        feature_importance_final=model.feature_importances_
        importance_dict = dict(zip(feature_importance_final, feature_names))
        importance_tuplelist = [(abs(value), feature) for (value, feature) in importance_dict.items()]
        importance_sorted = sorted(importance_tuplelist, reverse=True)
        x_feature = [j for (i, j) in importance_sorted]
        y_importance_value = [i for (i, j) in importance_sorted]
        print(f'{model_name}_tree_feature_names(top15):')
        for i in x_feature[:15]:
            print(i)
        print(f'average_{model_name}_feature_values(top15):')
        for i in y_importance_value[:15]:
            print(i)
        
        #记录更优的分数
        best_r2_score=current_r2_score
        best_mae_score=np.mean(mae_scores)
        best_mape_score=np.mean(mape_scores)
        if save_model:
            joblib.dump(model,   f'models/{model_name}/{model_name}_best_new.pkl')
            print(f'保存更优{model_name}模型文件') 
        print('平均 R2 分数:', best_r2_score)
        print('平均 MAPE 分数:', best_mape_score * 100)  # 转换为百分比
        print('平均MAE分数:',best_mae_score)
        

    
        
"""
打印并输出分数至表格
"""
df = pd.read_excel('score_different_algorithms.xlsx')
new_data = {'Model': f'{model_name}', 'R^2': best_r2_score , 'MAPE': best_mape_score * 100, 'MAE': best_mae_score}
new_data_df = pd.DataFrame([new_data])
# 将新数据添加到数据框中
df = pd.concat([df, new_data_df], ignore_index=True)
# 保存数据框到Excel表格
df.to_excel('score_different_algorithms.xlsx', index=False)
print(f'{model_name}算法的best_epoch=',best_epoch)

文件夹models/gboost已存在
The file 'score_different_algorithms.xlsx' exists.
Index(['MagpieData minimum Number', 'MagpieData maximum Number',
       'MagpieData range Number', 'MagpieData mean Number',
       'MagpieData avg_dev Number', 'MagpieData mode Number',
       'MagpieData minimum MendeleevNumber',
       'MagpieData maximum MendeleevNumber',
       'MagpieData range MendeleevNumber', 'MagpieData mean MendeleevNumber',
       ...
       'avg f valence electrons', 'frac s valence electrons',
       'frac p valence electrons', 'frac d valence electrons',
       'frac f valence electrons', 'Grain Size', 'Habit Plane',
       'Calculated Solid Solution', 'Calculated Grain Boundary',
       'Calculated Yield Strength'],
      dtype='object', length=277)
gboost_tree_feature_names(top15):
Grain Size
Interant electrons
Habit Plane
Calculated Grain Boundary
MagpieData mean GSvolume_pa
Zr fraction
Interant d electrons
Calculated Yield Strength
MagpieData avg_dev NValence
Yang delta
Ca fractio

In [14]:
from sklearn.ensemble import AdaBoostRegressor
import numpy as np
from sklearn.datasets import make_regression
import pandas as pd
from sklearn.model_selection import StratifiedKFold,train_test_split,cross_validate,cross_val_score,KFold
from sklearn.metrics import r2_score, mean_absolute_percentage_error,mean_absolute_error,make_scorer
from utils import create_empty_ls,create_dir,mycopyfile,mycopydir
import joblib
from openpyxl import load_workbook
import os
"""
路径的配置和选择
"""
model_name='adaboost'

save_model=True
del_0qf=True #是否删除0屈服强度数据
main_path=f'models/{model_name}' # 这里是模型所在的文件位置
create_dir(main_path,is_mainpath=True)
# excel=f'index/{model_name}_index.xlsx'
# if os.path.exists(excel):
#     print(f"The file '{excel}' exists.")
# else:
#     print(f"The file '{excel}' does not exist.")

# 如果分数表格不存在就新建一个
score_path='score_different_algorithms.xlsx'
if os.path.exists(score_path):
    print(f"The file '{score_path}' exists.")
else:
    print(f"The file '{score_path}' does not exist.")
    df = pd.DataFrame(columns=['Model', 'R^2', 'MAPE', 'MAE'])
    df.to_excel('score_different_algorithms.xlsx', index=False)


"""
文件读取部分
"""
# 读取 Excel 文件
df = pd.read_excel('train_set.xlsx', index_col=0)  # 引入这一列之后，原本的第一列就是一个索引，第0列会从有意义的列开始
# 删除包含空值的行
if del_0qf:
    df = df[df['屈服强度'] != 0].reset_index(drop=True)
np.random.seed(200) #设置随机数种子
feature_names = df.drop(columns=['Precipitate Distribution', '屈服强度', '抗拉强度 (UTS)', '追踪编号']).columns 
print(feature_names)
X = df.drop(columns=['Precipitate Distribution', '屈服强度', '抗拉强度 (UTS)', '追踪编号'])  # 特征: 最后四列之前的所有列
y = df['抗拉强度 (UTS)']
"""
设置初始化
"""
n_splits=5
# 初始化StratifiedKFold
# 初始化XGBoost回归器
# model = XGBRegressor(random_state=200,gamma=0.2)# 设置随机种子以确保结果的可重复性
# 初始化列表以存储每个折叠的分数
# 定义评估指标
scoring = {
    'r2':make_scorer(r2_score,greater_is_better=True),
    'MAE': make_scorer(mean_absolute_error, greater_is_better=False),
    'MAPE': make_scorer(mean_absolute_percentage_error, greater_is_better=False)
}
r2_scores_ls = []
mape_scores = []
mae_scores=[]
best_epoch=0
best_r2_score = float('-inf')
best_mae_score= float('-inf')
best_mape_score= float('-inf')
ls1=['fold'+str(i) for i in range(1,n_splits+1)]
fold_dict=dict(zip(ls1,[0 for i in range(0,len(ls1))]))

num_each_fold_train=np.floor((np.int32(X.shape[0])-1)*(n_splits-1)/n_splits)
num_each_fold_test=(np.int32(X.shape[0])-1)//n_splits

"""
模型训练部分,根据交叉验证结果评估模型性能
"""
epoches=1
for epoch in range(epoches):
    kf = KFold(n_splits=5, shuffle=True, random_state=0+10*epoch)# 设置随机种子以确保结果的可重复性
    model = AdaBoostRegressor(random_state=42, n_estimators=100)
    model.fit(X_train, y_train)
    
    # 计算分折交叉验证结果
    results = cross_validate(model,X,y,cv=kf, scoring=scoring, return_train_score=False)
    # 输出分折交叉验证结果
    for i, (r2, mae, mape) in enumerate(zip(results['test_r2'],results['test_MAE'], results['test_MAPE']), start=1):
        # 记录不同折的分数结果
        r2_scores_ls.append(r2)
        mape_scores.append(-mape)
        mae_scores.append(-mae)
        # print(f"折 {i}: r2={r2:.3f},MAE = {-mae:.3f}, MAPE = {-mape:.3f}")

    current_r2_score=np.mean(r2_scores_ls)
    # scores = cross_val_score(model, X, y_binned, cv=skf, scoring=make_scorer(r2_score))
    # print('r2_scores:',scores)
    # current_r2_score=scores.mean()
    # 只对更优的结果进行保存
    if current_r2_score>best_r2_score:
        best_epoch=epoch
        # 记录模型的特征重要性
        feature_importance_final=model.feature_importances_
        importance_dict = dict(zip(feature_importance_final, feature_names))
        importance_tuplelist = [(abs(value), feature) for (value, feature) in importance_dict.items()]
        importance_sorted = sorted(importance_tuplelist, reverse=True)
        x_feature = [j for (i, j) in importance_sorted]
        y_importance_value = [i for (i, j) in importance_sorted]
        print(f'{model_name}_tree_feature_names(top15):')
        for i in x_feature[:15]:
            print(i)
        print(f'average_{model_name}_feature_values(top15):')
        for i in y_importance_value[:15]:
            print(i)
        
        #记录更优的分数
        best_r2_score=current_r2_score
        best_mae_score=np.mean(mae_scores)
        best_mape_score=np.mean(mape_scores)
        if save_model:
            joblib.dump(model,   f'models/{model_name}/{model_name}_best_new.pkl')
            print(f'保存更优{model_name}模型文件') 
        print('平均 R2 分数:', best_r2_score)
        print('平均 MAPE 分数:', best_mape_score * 100)  # 转换为百分比
        print('平均MAE分数:',best_mae_score)
        

    
        
"""
打印并输出分数至表格
"""

df = pd.read_excel('score_different_algorithms.xlsx')
new_data = {'Model': f'{model_name}', 'R^2': best_r2_score , 'MAPE': best_mape_score * 100, 'MAE': best_mae_score}
new_data_df = pd.DataFrame([new_data])
# 将新数据添加到数据框中
df = pd.concat([df, new_data_df], ignore_index=True)
# 保存数据框到Excel表格
df.to_excel('score_different_algorithms.xlsx', index=False)
print(f'{model_name}算法的best_epoch=',best_epoch)

文件夹models/adaboost已存在
The file 'score_different_algorithms.xlsx' exists.
Index(['MagpieData minimum Number', 'MagpieData maximum Number',
       'MagpieData range Number', 'MagpieData mean Number',
       'MagpieData avg_dev Number', 'MagpieData mode Number',
       'MagpieData minimum MendeleevNumber',
       'MagpieData maximum MendeleevNumber',
       'MagpieData range MendeleevNumber', 'MagpieData mean MendeleevNumber',
       ...
       'avg f valence electrons', 'frac s valence electrons',
       'frac p valence electrons', 'frac d valence electrons',
       'frac f valence electrons', 'Grain Size', 'Habit Plane',
       'Calculated Solid Solution', 'Calculated Grain Boundary',
       'Calculated Yield Strength'],
      dtype='object', length=280)
adaboost_tree_feature_names(top15):
Habit Plane
Grain Size
frac p valence electrons
VEC mean
Interant electrons
frac d valence electrons
Electronegativity delta
MagpieData minimum GSvolume_pa
Kr fraction
Configuration entropy
Rb fractio

In [15]:
from sklearn.neighbors import KNeighborsRegressor 
from sklearn.inspection import permutation_importance
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold,train_test_split,cross_validate,cross_val_score,KFold
from sklearn.metrics import r2_score, mean_absolute_percentage_error,mean_absolute_error,make_scorer
from utils import create_empty_ls,create_dir,mycopyfile,mycopydir
import joblib
from openpyxl import load_workbook
import os
"""
路径的配置和选择
"""
model_name='knn'

save_model=True
del_0qf=True #是否删除0屈服强度数据
main_path=f'models/{model_name}' # 这里是模型所在的文件位置
create_dir(main_path,is_mainpath=True)
# excel=f'index/{model_name}_index.xlsx'
# if os.path.exists(excel):
#     print(f"The file '{excel}' exists.")
# else:
#     print(f"The file '{excel}' does not exist.")

# 如果分数表格不存在就新建一个
score_path='score_different_algorithms.xlsx'
if os.path.exists(score_path):
    print(f"The file '{score_path}' exists.")
else:
    print(f"The file '{score_path}' does not exist.")
    df = pd.DataFrame(columns=['Model', 'R^2', 'MAPE', 'MAE'])
    df.to_excel('score_different_algorithms.xlsx', index=False)


"""
文件读取部分
"""
# 读取 Excel 文件
df = pd.read_excel('train_set_new.xlsx', index_col=0)  # 引入这一列之后，原本的第一列就是一个索引，第0列会从有意义的列开始
# 删除包含空值的行
if del_0qf:
    df = df[df['屈服强度'] != 0].reset_index(drop=True)
np.random.seed(200) #设置随机数种子
feature_names = df.drop(columns=['Precipitate Distribution', '屈服强度', '抗拉强度 (UTS)', '追踪编号']).columns 
print(feature_names)
X = df.drop(columns=['Precipitate Distribution', '屈服强度', '抗拉强度 (UTS)', '追踪编号'])  # 特征: 最后四列之前的所有列
y = df['抗拉强度 (UTS)']
"""
设置初始化
"""
n_splits=5
# 初始化StratifiedKFold
# 初始化XGBoost回归器
# model = XGBRegressor(random_state=200,gamma=0.2)# 设置随机种子以确保结果的可重复性
# 初始化列表以存储每个折叠的分数
# 定义评估指标
scoring = {
    'r2':make_scorer(r2_score,greater_is_better=True),
    'MAE': make_scorer(mean_absolute_error, greater_is_better=False),
    'MAPE': make_scorer(mean_absolute_percentage_error, greater_is_better=False)
}
r2_scores_ls = []
mape_scores = []
mae_scores=[]
best_epoch=0
best_r2_score = float('-inf')
best_mae_score= float('-inf')
best_mape_score= float('-inf')
ls1=['fold'+str(i) for i in range(1,n_splits+1)]
fold_dict=dict(zip(ls1,[0 for i in range(0,len(ls1))]))

num_each_fold_train=np.floor((np.int32(X.shape[0])-1)*(n_splits-1)/n_splits)
num_each_fold_test=(np.int32(X.shape[0])-1)//n_splits

"""
模型训练部分,根据交叉验证结果评估模型性能
"""
epoches=1
for epoch in range(epoches):
    kf = KFold(n_splits=5, shuffle=True, random_state=0+10*epoch)# 注意这里修改了random_state以确保每次初始化不同
    model = KNeighborsRegressor(n_neighbors=5) 
    model.fit(X_train, y_train)
    
    # 计算分折交叉验证结果
    results = cross_validate(model,X,y,cv=kf, scoring=scoring, return_train_score=False)
    # 输出分折交叉验证结果
    for i, (r2, mae, mape) in enumerate(zip(results['test_r2'],results['test_MAE'], results['test_MAPE']), start=1):
        # 记录不同折的分数结果
        r2_scores_ls.append(r2)
        mape_scores.append(-mape)
        mae_scores.append(-mae)
        # print(f"折 {i}: r2={r2:.3f},MAE = {-mae:.3f}, MAPE = {-mape:.3f}")

    current_r2_score=np.mean(r2_scores_ls)
    # scores = cross_val_score(model, X, y_binned, cv=skf, scoring=make_scorer(r2_score))
    # print('r2_scores:',scores)
    # current_r2_score=scores.mean()
    # 只对更优的结果进行保存
    print(current_r2_score)
    if current_r2_score>best_r2_score:
        # 记录模型的特征重要性
        # perm_importance = permutation_importance(model, X_test, y_test, n_repeats=30, random_state=42)
        # # 获取特征重要性值
        # importances = perm_importance.importances_mean
        # 
        # # 对特征重要性进行排序
        # sorted_idx = np.argsort(importances)[::-1]
        # 
        # print(f'{model_name}_feature_names(top15):')
        # for i in feature_names[sorted_idx[:15]]:
        #     print(i)
        # print(f'average_{model_name}_feature_values(top15):')
        # for i in importances[sorted_idx[:15]]:
        #     print(i)
        
        
        #记录更优的分数
        best_epoch=epoch
        best_r2_score=current_r2_score
        best_mae_score=np.mean(mae_scores)
        best_mape_score=np.mean(mape_scores)
        if save_model:
            joblib.dump(model,   f'models/{model_name}/{model_name}_best_new.pkl')
            print(f'保存更优{model_name}模型文件') 
        print('平均 R2 分数:', best_r2_score)
        print('平均 MAPE 分数:', best_mape_score * 100)  # 转换为百分比
        print('平均MAE分数:',best_mae_score)
          
"""
打印并输出分数至表格
"""

df = pd.read_excel('score_different_algorithms.xlsx')
new_data = {'Model': f'{model_name}', 'R^2': best_r2_score , 'MAPE': best_mape_score * 100, 'MAE': best_mae_score}
new_data_df = pd.DataFrame([new_data])
# 将新数据添加到数据框中
df = pd.concat([df, new_data_df], ignore_index=True)
# 保存数据框到Excel表格
df.to_excel('score_different_algorithms.xlsx', index=False)
print(f'{model_name}算法的best_epoch=',best_epoch)





文件夹models/knn已存在
The file 'score_different_algorithms.xlsx' exists.
Index(['MagpieData minimum Number', 'MagpieData maximum Number',
       'MagpieData range Number', 'MagpieData mean Number',
       'MagpieData avg_dev Number', 'MagpieData mode Number',
       'MagpieData minimum MendeleevNumber',
       'MagpieData maximum MendeleevNumber',
       'MagpieData range MendeleevNumber', 'MagpieData mean MendeleevNumber',
       ...
       'avg f valence electrons', 'frac s valence electrons',
       'frac p valence electrons', 'frac d valence electrons',
       'frac f valence electrons', 'Grain Size', 'Habit Plane',
       'Calculated Solid Solution', 'Calculated Grain Boundary',
       'Calculated Yield Strength'],
      dtype='object', length=277)
0.49376691799242123
保存更优knn模型文件
平均 R2 分数: 0.49376691799242123
平均 MAPE 分数: 17.269279596224898
平均MAE分数: 42.48753234050377
knn算法的best_epoch= 0


In [16]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error, make_scorer
import pandas as pd
import numpy as np

# 读取数据
df = pd.read_excel('train_set_new.xlsx', index_col=0)

# 删除屈服强度为0的行
df = df[df['屈服强度'] != 0].reset_index(drop=True)
display(df)
# 设置随机种子
np.random.seed(200)

# 定义特征和目标变量
X = df.drop(columns=['Precipitate Distribution', '屈服强度', '抗拉强度 (UTS)', '追踪编号'])  # 特征: 最后四列之前的所有列
y = df['抗拉强度 (UTS)']
# 划分数据集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=60)

# 初始化KNN回归模型
model = KNeighborsRegressor(n_neighbors=5)

# 训练模型
model.fit(X_train, y_train)

# 预测测试集
y_pred = model.predict(X_test)

# 计算评估指标
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
mape = mean_absolute_percentage_error(y_test, y_pred)

print(f'R^2 Score: {r2:.4f}')
print(f'Mean Absolute Error (MAE): {mae:.4f}')
print(f'Mean Absolute Percentage Error (MAPE): {mape:.4f}')

# 使用KFold交叉验证来评估模型性能
kf = KFold(n_splits=5, shuffle=True, random_state=60)
scoring = {
    'r2': 'r2',
    'MAE': make_scorer(mean_absolute_error),
    'MAPE': make_scorer(mean_absolute_percentage_error)
}
results = cross_val_score(model, X, y, cv=kf, scoring=make_scorer(r2_score))
print(f'Cross-validated R^2 scores: {results}')
print(f'Average R^2 score from CV: {np.mean(results):.4f}')

,MagpieData minimum Number,MagpieData maximum Number,MagpieData range Number,MagpieData mean Number,MagpieData avg_dev Number,MagpieData mode Number,MagpieData minimum MendeleevNumber,MagpieData maximum MendeleevNumber,MagpieData range MendeleevNumber,MagpieData mean MendeleevNumber,...,frac f valence electrons,Grain Size,Precipitate Distribution,Habit Plane,Calculated Solid Solution,Calculated Grain Boundary,Calculated Yield Strength,屈服强度,抗拉强度 (UTS),追踪编号
0,12,30,18,12.131276,0.247009,12,52,73,21,68.266826,...,0.000000,4.00,0,0,17.669989,194.321983,226.991971,364.0,393.0,No630-6Al-1Zn-0.15Mn -3
1,12,64,52,12.922815,1.811506,12,23,68,45,67.259700,...,0.054984,36.00,3,1,57.602249,48.504397,121.106646,125.0,200.0,No232-Mg-9Gd-1Sm-0.5Zr-2
2,12,64,52,13.780587,3.401587,12,12,69,57,66.376359,...,0.078055,50.00,3,5,94.613373,69.789053,179.402426,200.0,340.0,No16-Mg–14Gd–3Y–1.8Zn–0.5Zr -9
3,12,64,52,13.179217,2.292508,12,12,68,56,66.814372,...,0.052520,1.50,2,1,76.138523,262.520047,353.658570,290.0,280.0,No96-Mg-9.3Gd-2.7Y-0.65Ag-0.52Zr-0.18Ce -4
4,12,30,18,12.112392,0.217496,12,52,73,21,68.118900,...,0.000000,15.00,0,0,11.604694,86.484728,113.089422,148.0,280.0,No460-3Al-1Zn-0.3Mn -22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
734,12,38,26,12.053936,0.104733,12,8,73,65,68.099036,...,0.000000,130.00,1,1,12.020723,23.509472,50.530195,58.5,220.0,No744-Mg-3Al-0.4Mn-0.05Sr
735,12,64,52,12.937303,1.825661,12,12,69,57,67.051202,...,0.033095,41.77,3,5,61.765688,53.715298,130.480986,223.0,295.0,No226-Mg-6Gd-3Y-1Zn-0.4Zr-0.7Ag-3
736,12,30,18,12.152650,0.281635,12,7,73,66,67.973850,...,0.000000,30.00,3,2,27.156102,71.029741,113.185842,91.1,144.3,No312-Mg-7.6Al-0.5Zn-1Ca-2
737,12,64,52,12.126080,0.237903,12,12,73,61,68.188783,...,0.003189,52.10,1,1,23.809563,45.036179,83.845741,125.0,217.0,No383-Mg-6Al-Zn-0.3Y-0.6Gd


R^2 Score: 0.4704
Mean Absolute Error (MAE): 46.3700
Mean Absolute Percentage Error (MAPE): 0.2054
Cross-validated R^2 scores: [0.47144224 0.4734247  0.45109753 0.50376077 0.53507098]
Average R^2 score from CV: 0.4870


In [17]:
# from sklearn.neighbors import KNeighborsRegressor
# import numpy as np
# 
# # 创建一些简单的数据
# X_train = np.array([[1], [2], [3], [4], [5]])
# y_train = np.array([1, 2, 3, 4, 5])
# X_test = np.array([[1.5], [2.5]])
# 
# # 初始化并训练模型
# model = KNeighborsRegressor(n_neighbors=2)
# model.fit(X_train, y_train)
# 
# # 预测
# print(model.predict(X_test))
